# Prebuilt middleware  预构建中间件
## Provider-agnostic middleware 与提供商无关的中间件
以下中间件可与任何 LLM 提供商配合使用：

### Summarization  总结
当接近消息数量上限时，自动生成对话历史记录摘要，保留最近的消息，同时压缩较早的上下文。摘要功能适用于以下情况：
- 持续时间过长的对话，超出上下文窗口。
- 多轮对话，历史悠久。
- 需要保留完整对话上下文的应用场景。

In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()

base_url = os.getenv("QWEN_BASE_URL")
api_key = os.getenv("QWEN_API_KEY")

model = ChatOpenAI(
    model="glm-4.7",
    base_url=base_url,
    api_key=api_key,
)

config参数:
- **model**  模型 string | BaseChatModel (必填) 用于生成摘要的模型
- **trigger**  触发器 ContextSize | list[ContextSize] | None 触发摘要生成的条件。
    - 单个 ContextSize 元组（必须满足指定条件）
    - ContextSize 元组列表（必须满足任意条件 - 逻辑 OR）
    - 条件应为以下之一：
        - fraction （浮点数）：模型上下文大小的比例（0-1）
        - tokens （整数）：绝对标记计数
        - messages (int): 消息计数
    > 必须至少指定一个条件。如果未提供条件，则不会自动触发摘要功能。
- **keep**  ContextSizedefault:"('messages', 20)" 摘要后要保留多少上下文信息？请指定以下选项之一：
    - fraction （浮点数）：要保留的模型上下文大小的比例（0-1）
    - tokens （整数）：要保留的令牌绝对数量
    - messages (int): 要保留的最新消息数量
- **token_counter**  令牌计数器 function 自定义词元计数函数。默认按字符计数。
- **summary_prompt**  摘要提示 string  用于摘要的自定义提示模板。如果未指定，则使用内置模板。模板应包含 {messages} 占位符，用于插入对话历史记录。
- **trim_tokens_to_summarize** number default:"4000" 生成摘要时要包含的最大令牌数。消息在生成摘要之前会被截断以符合此限制。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware


# Single condition: trigger if tokens >= 4000
agent = create_agent(
    model,
    # tools=[your_weather_tool, your_calculator_tool],
    middleware=[
        SummarizationMiddleware(
            model,
            trigger=("tokens", 4000),
            keep=("messages", 20),
        ),
    ],
)

# Multiple conditions: trigger if number of tokens >= 3000 OR messages >= 6
agent2 = create_agent(
    model,
    # tools=[your_weather_tool, your_calculator_tool],
    middleware=[
        SummarizationMiddleware(
            model,
            trigger=[
                ("tokens", 3000),
                ("messages", 6),
            ],
            keep=("messages", 20),
        ),
    ],
)

# Using fractional limits
agent3 = create_agent(
    model,
    # tools=[your_weather_tool, your_calculator_tool],
    middleware=[
        SummarizationMiddleware(
            model,
            trigger=("fraction", 0.8),
            keep=("fraction", 0.3),
        ),
    ],
)

### Human-in-the-loop  人机交互
在工具调用执行前，暂停代理程序的执行，以便人工审批、编辑或拒绝这些调用。 人机协作机制在以下情况下非常有用：
- 需要人工批准的高风险操作（例如数据库写入、金融交易）。
- 需要人工监督的合规工作流程。
- 长时间的对话，其中人类的反馈会指导智能体。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def your_read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def your_send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model,
    tools=[your_read_email_tool, your_send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "your_send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "your_read_email_tool": False,
            }
        ),
    ],
)

### Model call limit  模型调用限制
限制模型调用次数以防止无限循环或过高的成本。模型调用次数限制在以下情况下非常有用：
- 防止失控代理发出过多 API 调用。
- 对生产部署实施成本控制。
- 在特定通话预算范围内测试客服人员的行为。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model,
    checkpointer=InMemorySaver(),  # Required for thread limiting
    tools=[],
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=10,
            run_limit=5,
            exit_behavior="end",
        ),
    ],
)

config:
- **thread_limit**  线程限制 `number` 线程中所有运行的总模型调用次数上限。默认无限制。
- **run_limit**  运行限制 `number` 单次调用中模型调用次数的上限。默认无限制。
- **exit_behavior**  退出行为 string `default:"end"` 达到限制时的行为。选项： 'end' （正常终止）或 'error' （引发异常）。

### Tool call limit  工具调用限制
通过限制工具调用次数来控制代理的执行，可以全局限制所有工具，也可以限制特定工具。工具调用次数限制在以下情况下非常有用：
- 防止过度调用昂贵的外部 API。
- 限制网络搜索或数据库查询。
- 对特定工具的使用频率实施限制。
- 防止代理程序陷入失控循环。
Configuration options :
- **tool_name**  工具名称 `string` 要限制的具体工具名称。如果未提供，则限制将应用于所有工具。
- **thread_limit**  线程限制 `number` 线程（会话）内所有运行中工具调用的最大次数。此值在具有相同线程 ID 的多次调用中保持不变。需要检查点来维护状态。 None 表示无线程限制。
- **run_limit**  运行限制 `number` 每次调用（一个用户消息→响应周期）的最大工具调用次数。每次收到新的用户消息时都会重置。如果设置为 None ，则表示没有运行次数限制。
> 注意： thread_limit 或 run_limit 中至少要指定一个。
- **exit_behavior**  退出行为 `string` `default:"continue"` 达到限制时的行为：
    - `continue` （默认）——阻止超出限制的工具调用，并显示错误消息，允许其他工具和模型继续运行。模型会根据错误消息决定何时结束。
    - `error` - 引发 ToolCallLimitExceededError 异常，立即停止执行
    - `end` - 立即停止执行，并向 ToolMessage 和 AI 发送消息，说明超出了刀具调用次数的限制。仅在限制单个刀具时有效；如果其他刀具有待处理的调用，则会引发 NotImplementedError 。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware


global_limiter = ToolCallLimitMiddleware(thread_limit=20, run_limit=10)
search_limiter = ToolCallLimitMiddleware(tool_name="search", thread_limit=5, run_limit=3)
database_limiter = ToolCallLimitMiddleware(tool_name="query_database", thread_limit=10)
strict_limiter = ToolCallLimitMiddleware(tool_name="scrape_webpage", run_limit=2, exit_behavior="error")

agent = create_agent(
    model="gpt-4.1",
    # tools=[search_tool, database_tool, scraper_tool],
    middleware=[global_limiter, search_limiter, database_limiter, strict_limiter],
)

### Model fallback  模型回退
当主模型失效时，自动回退到备用模型。模型回退功能适用于以下情况：
- 构建能够处理模型故障的弹性代理。
- 通过采用更便宜的型号来优化成本。
- OpenAI、Anthropic 等供应商之间的冗余。
Configuration options:
- **first_model**  string | BaseChatModel `required` 当主模型失效时，要尝试的第一个备用模型。可以是模型标识符字符串（例如， 'openai:gpt-4.1-mini' ），也可以是 BaseChatModel 实例。
- ***additional_models**  string | BaseChatModel 如果之前的模型失败，则按顺序尝试其他备用模型

### PII detection  PII 检测
使用可配置策略检测和处理对话中的个人身份信息 (PII)。PII 检测可用于以下用途：
- 医疗保健和金融应用，需满足合规性要求。
- 需要清理日志的客服人员。
- 任何处理敏感用户数据的应用程序。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model,
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
    ],
)

#### Custom PII types  自定义 PII 类型
可以通过提供 detector 参数来创建自定义 PII 类型。

创建自定义检测器的三种方法：
- 正则表达式模式字符串 - 简单模式匹配
- 自定义功能 - 带有验证功能的复杂检测逻辑
- Custom detector function signature 自定义检测器函数签名：

    检测器函数必须接受一个字符串（内容）并返回匹配项：

    返回一个字典列表，字典包含 text 、 start 和 end 三个键：
```python
def detector(content: str) -> list[dict[str, str | int]]:
    return [
        {"text": "matched_text", "start": 0, "end": 12},
        # ... more matches
    ]
```
- 对于简单的模式，请使用正则表达式字符串。
- 需要添加标志（例如，不区分大小写的匹配）时，请使用正则表达式对象。
- 需要除模式匹配之外的验证逻辑时，请使用自定义函数。
- 自定义函数让您可以完全控制检测逻辑，并实现复杂的验证规则。

Configuration options :
- **pii_type** string `required` 要检测的个人身份信息类型。可以是内置类型（ email 、 credit_card 、 ip 、 mac_address 、 url ）或自定义类型名称。
- **strategy** string `default:"redact"` 如何处理检测到的个人身份信息？选项：
    - `block` - 检测到时引发异常
    - `redact` - 替换为 [REDACTED_{PII_TYPE}]
    - `mask` - 部分掩蔽（例如， ****-****-****-1234 ）
    - `hash` - 替换为确定性哈希
- **detector** function | regex 自定义检测器函数或正则表达式模式。如果未提供，则使用内置的 PII 类型检测器。
- **apply_to_input** boolean `default:"True"` 在模型调用之前检查用户消息
- **apply_to_output** boolean `default:"False"` 模型调用后检查 AI 消息
- **apply_to_tool_results** boolean `default:"False"` 执行后检查工具结果消息

### To-do list  待办事项清单
为代理配备任务规划和跟踪功能，以处理复杂的多步骤任务。待办事项清单在以下方面非常有用：
- 需要协调使用多种工具的复杂多步骤任务。
- 需要长期关注项目进展情况的项目。
> 该中间件会自动为代理提​​供 write_todos 工具和系统提示，以指导有效的任务规划。
Configuration options :
- **system_prompt**  系统提示符 `string` 自定义系统提示，用于指导待办事项的使用。如果未指定，则使用内置提示。
- **tool_description**  工具描述 `string` 为 write_todos 工具自定义描述。如果未指定，则使用内置描述。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware

agent = create_agent(
    model="gpt-4.1",
    # tools=[read_file, write_file, run_tests],
    middleware=[TodoListMiddleware()],
)

### LLM tool selector  LLM 工具选择器
使用 LLM 在调用主模型之前智能地选择相关工具。LLM 工具选择器可用于以下用途：
- 拥有众多工具（10 个以上）的代理，但大多数工具与每次查询都不相关。
- 通过过滤不相关的工具来减少令牌使用量。
- 提高模型聚焦性和准确性。
> 该中间件使用结构化输出向 LLM 询问哪些工具与当前查询最相关。结构化输出模式定义了可用工具的名称和描述。模型提供者通常会在后台将这些结构化输出信息添加到系统提示中。

Configuration options:
- **model** `string | BaseChatModel` 用于工具选择的模型。默认使用代理的主模型。
- **system_prompt** `string` 选择模型的说明。如果未指定，则使用内置提示。
- **max_tools** `number` 要选择的工具的最大数量。如果模型选择的工具数量超过 max_tools，则只会使用前 max_tools 个工具。如果未指定，则无限制。
- **always_include** `list[string]` 无论选择哪个选项，始终包含的工具名称。这些工具名称不计入 max_tools 限制。

### Tool retry  工具重试
使用可配置的指数退避算法自动重试失败的工具调用。工具重试功能适用于以下情况：
- 处理外部 API 调用中的瞬态故障。
- 提高网络依赖型工具的可靠性。
- 构建能够优雅地处理临时错误的弹性代理。
Configuration options:
- **max_retries** number `default:"3"` 首次调用后的最大重试次数（默认值为 3 次）
- **tools** `list[BaseTool | str]` 可选的工具列表或工具名称，用于指定要应用重试逻辑的工具。如果为 None ，则应用于所有工具。
- **retry_on** `tuple[type[Exception], ...] | callabledefault:"(Exception,)"`可以是要重试的异常类型元组，也可以是接受异常并返回 True 的可调用对象(如果应该重试则返回 True)。
- **on_failure** `string | callable` `default:"return_message"`当所有重试次数都用尽时的行为。选项：
    - `return_message` - 返回包含错误详情 ToolMessage （允许 LLM 处理故障）
    - `raise` - 重新引发异常（停止代理执行）
    - 自定义可调用对象 - 该函数接收异常并返回 ToolMessage 内容的字符串。
- **backoff_factor** `number` `default:"2.0"`指数退避的乘数。每次重试等待 initial_delay * (backoff_factor ** retry_number) 秒。设置为 0.0 表示恒定延迟。
- **initial_delay**  初始延迟 `number` `default:"1.0"`首次重试前的初始延迟时间（秒）
- **max_delay**  最大延迟 `number` `default:"60.0"` 重试之间的最大延迟时间（以秒为单位）（限制指数级退避增长）
- **jitter**  `boolean` `default:"true"` 是否在延迟中加入随机抖动（ ±25% ）以避免群体效应

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware


agent = create_agent(
    model="gpt-4.1",
    # tools=[search_tool, database_tool, api_tool],
    middleware=[
        ToolRetryMiddleware(
            max_retries=3,# 设置最大重试次数为3
            backoff_factor=2.0,# 每次重试的等待时间将是前一次的2倍
            initial_delay=1.0,# 初始等待时间为1秒
            max_delay=60.0,# 最大等待时间为60秒
            jitter=True,# 在每次重试的等待时间上添加随机抖动，防止多个请求同时重试导致的冲击
            tools=["api_tool"],# 仅针对api_tool启用重试机制
            retry_on=(ConnectionError, TimeoutError),# 仅在遇到ConnectionError或TimeoutError时触发重试
            on_failure="continue",# 遇到重试失败时继续执行后续操作而不是抛出异常
        ),
    ],
)

### Model retry  模型重试
使用可配置的指数退避算法自动重试失败的模型调用。模型重试功能适用于以下情况：
- 处理模型 API 调用中的瞬态故障。
- 提高网络相关模型请求的可靠性。
- 构建能够优雅地处理临时模型错误的弹性代理。
Configuration options :
- **max_retries** `number` `default:"3"` 首次调用后的最大重试次数（默认值为 3 次）
- **retry_on** `tuple[type[Exception], ...] | callabledefault:"(Exception,)"` 可以是要重试的异常类型元组，也可以是接受异常并返回 True 的可调用对象（如果应该重试则返回 True 。
- **on_failure** `string | callable` `default:"continue"`当所有重试次数都用尽时的行为。选项：
    - `continue` （默认）- 返回包含错误详情的 AIMessage ，允许代理优雅地处理故障。
    - `error` - 重新引发异常（停止代理执行）
    - 自定义可调用对象 - 接受异常并返回 AIMessage 内容字符串的函数
- **backoff_factor** `number` `default:"2.0"` 指数退避的乘数。每次重试等待 initial_delay * (backoff_factor ** retry_number) 秒。设置为 0.0 表示恒定延迟。
- **initial_delay**  初始延迟 `number` `default:"1.0"`首次重试前的初始延迟时间（秒）
- **max_delay**  最大延迟 `number` `default:"60.0"` 重试之间的最大延迟时间（以秒为单位）（限制指数级退避增长）
- **jitter**  `boolean` `default:"true"` 是否在延迟中加入随机抖动（ ±25% ）以避免群体效应

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware


# Basic usage with default settings (2 retries, exponential backoff)
agent = create_agent(
    model="gpt-4.1",
    # tools=[search_tool],
    middleware=[ModelRetryMiddleware()],
)

# Custom exception filtering
class TimeoutError(Exception):
    """Custom exception for timeout errors."""
    pass

class ConnectionError(Exception):
    """Custom exception for connection errors."""
    pass

# 重试最多4次，仅在遇到TimeoutError或ConnectionError时触发，重试间隔为1.5倍指数增长
retry = ModelRetryMiddleware(
    max_retries=4,
    retry_on=(TimeoutError, ConnectionError),
    backoff_factor=1.5,
)


def should_retry(error: Exception) -> bool:
    # Only retry on rate limit errors
    if isinstance(error, TimeoutError):
        return True
    # Or check for specific HTTP status codes
    if hasattr(error, "status_code"):
        return error.status_code in (429, 503)
    return False

# 自定义重试条件函数，根据错误类型或属性决定是否重试
retry_with_filter = ModelRetryMiddleware(
    max_retries=3,
    retry_on=should_retry,
)

# 返回错误消息而不是抛出异常
retry_continue = ModelRetryMiddleware(
    max_retries=4,
    on_failure="continue",  # Return AIMessage with error instead of raising
)

# 自定义错误处理函数，返回格式化的错误消息
def format_error(error: Exception) -> str:
    return f"Model call failed: {error}. Please try again later."

retry_with_formatter = ModelRetryMiddleware(
    max_retries=4,
    on_failure=format_error,
)

constant_backoff = ModelRetryMiddleware(
    max_retries=5,
    backoff_factor=0.0, 
    initial_delay=2.0,  
)

# 收集错误信息并在最终失败时抛出异常
strict_retry = ModelRetryMiddleware(
    max_retries=2,
    on_failure="error",  # 最终抛出异常
)

### LLM tool emulator  LLM 工具模拟器
使用 LLM 模拟工具执行以进行测试，用 AI 生成的响应替换实际的工具调用。LLM 工具模拟器可用于以下用途：
- 无需使用实际工具即可测试代理行为。
- 当外部工具不可用或成本高昂时，开发代理。
- 在实际工具实施之前，先对代理工作流程进行原型设计。
Configuration options:
- **tools** `list[str | BaseTool]` 要模拟的工具名称列表（字符串）或 BaseTool 实例。如果为 None （默认值），则模拟所有工具。如果为空列表 [] ，则不模拟任何工具。如果为包含工具名称/实例的数组，则仅模拟这些工具。
- **model** `string | BaseChatModel` 用于生成模拟工具响应的模型。可以是模型标识符字符串（例如'anthropic:claude-sonnet-4-6' ）或 BaseChatModel 实例。如果未指定，则默认为代理的模型。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolEmulator
from langchain.tools import tool


@tool
def get_weather(location: str) -> str:
    """Get the current weather for a location."""
    return f"Weather in {location}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email."""
    return "Email sent"


# Emulate all tools (default behavior)
agent = create_agent(
    model="gpt-4.1",
    tools=[get_weather, send_email],
    middleware=[LLMToolEmulator()],
)

# Emulate specific tools only
agent2 = create_agent(
    model="gpt-4.1",
    tools=[get_weather, send_email],
    middleware=[LLMToolEmulator(tools=["get_weather"])],
)

# Use custom model for emulation
agent4 = create_agent(
    model="gpt-4.1",
    tools=[get_weather, send_email],
    middleware=[LLMToolEmulator(model="claude-sonnet-4-6")],
)

### Context editing  上下文编辑
当达到令牌限制时，通过清除较旧的工具调用输出来管理对话上下文，同时保留最近的结果。这有助于在包含大量工具调用的长时间对话中保持上下文窗口的可控性。上下文编辑在以下情况下非常有用：
- 长时间的对话，期间调用了许多超出令牌限制的工具。
- 通过移除不再相关的旧工具输出来降低代币成本
- 仅保留上下文中最新的 N 个工具结果
Configuration options:
- **edits** `list[ContextEdit]` `default:"[ClearToolUsesEdit()]"` 要应用的 ContextEdit 策略列表
- **token_count_method** `string` `default:"approximate"`令牌计数方法。选项： 'approximate' 或 'model'
- **ClearToolUsesEdit** options:
    - **trigger** `number` `default:"100000"`触发编辑的令牌数量。当对话令牌数量超过此值时，较早的工具输出将被清除。
    - **clear_at_least** `number` `default:"0"`编辑运行时要回收的最小令牌数。如果设置为 0，则会根据需要清除任意数量的令牌。
    - **keep** `number` `default:"3"`必须保留的最新工具结果数量。这些结果将永远不会被清除。
    - **clear_tool_inputs** `boolean` `default:"False"`是否清除 AI 消息中原始工具调用参数。如果为 True ，则工具调用参数将被替换为空对象。
    - **exclude_tools** `list[string]` `default:"()"`要排除在清除范围之外的工具名称列表。这些工具的输出将永远不会被清除。
    - **placeholder** `string` `default:"[cleared]"`此处插入占位符文本，用于显示已清除的工具输出。此文本将替换原始工具消息内容。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ContextEditingMiddleware, ClearToolUsesEdit


agent = create_agent(
    model="gpt-4.1",
    # tools=[search_tool, your_calculator_tool, database_tool],
    middleware=[
        ContextEditingMiddleware(
            edits=[
                ClearToolUsesEdit(
                    trigger=2000,
                    keep=3,
                    clear_tool_inputs=False,
                    exclude_tools=[],
                    placeholder="[cleared]",
                ),
            ],
        ),
    ],
)

### Shell tool  Shell 工具
向代理公开持久化的 shell 会话以执行命令。Shell 工具中间件可用于以下用途：
- 需要执行系统命令的代理
- 开发和部署自动化任务
- 测试和验证工作流程
- 文件系统操作和脚本执行
Configuration options:
- **workspace_root**  `str | Path | None` shell 会话的基本目录。如果省略，则会在代理启动时创建一个临时目录，并在代理结束时删除该目录。
- **startup_commands** `tuple[str, ...] | list[str] | str | None`会话开始后按顺序执行的可选命令
- **shutdown_commands** `tuple[str, ...] | list[str] | str | None` 会话关闭前执行的可选命令
- **execution_policy** `BaseExecutionPolicy | None`执行策略控制超时、输出限制和资源配置。选项：
    - HostExecutionPolicy - 完全主机访问权限（默认）；最适合代理程序已在容器或虚拟机中运行的受信任环境。
    - DockerExecutionPolicy - 为每个代理运行启动一个独立的 Docker 容器，从而提供更强的隔离性
    - CodexSandboxExecutionPolicy - 重用 Codex CLI 沙箱以添加额外的系统调用/文件系统限制
- **redaction_rules** `tuple[RedactionRule, ...] | list[RedactionRule] | None`可选的编辑规则，用于在将命令输出返回给模型之前对其进行清理。(编辑规则在执行后应用，使用 HostExecutionPolicy 时无法阻止密钥或敏感数据的泄露。)
- **tool_description**  `str | None` 可选地覆盖已注册的 shell 工具描述 
- **shell_command**  `Sequence[str] | str | None`用于启动持久会话的可选 shell 可执行文件（字符串）或参数序列。默认为 /bin/bash 。
- **env**  `Mapping[str, Any] | None` 可选的环境变量，可提供给 shell 会话。这些值在命令执行前会被强制转换为字符串。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import (
    ShellToolMiddleware,
    HostExecutionPolicy,
    DockerExecutionPolicy,
    RedactionRule,
)


# Basic shell tool with host execution
agent = create_agent(
    model="gpt-4.1",
    # tools=[search_tool],
    middleware=[
        ShellToolMiddleware(
            workspace_root="/workspace",
            execution_policy=HostExecutionPolicy(),
        ),
    ],
)

# Docker isolation with startup commands
agent_docker = create_agent(
    model="gpt-4.1",
    tools=[],
    middleware=[
        ShellToolMiddleware(
            workspace_root="/workspace",
            startup_commands=["pip install requests", "export PYTHONPATH=/workspace"],
            execution_policy=DockerExecutionPolicy(
                image="python:3.11-slim",
                command_timeout=60.0,
            ),
        ),
    ],
)

# With output redaction (applied post execution)
agent_redacted = create_agent(
    model="gpt-4.1",
    tools=[],
    middleware=[
        ShellToolMiddleware(
            workspace_root="/workspace",
            redaction_rules=[
                RedactionRule(pii_type="api_key", detector=r"sk-[a-zA-Z0-9]{32}"),
            ],
        ),
    ],
)

### File search  文件搜索
在文件系统上提供 Glob 和 Grep 搜索工具。文件搜索中间件可用于以下用途：
- 代码探索与分析
- 按名称模式查找文件
- 使用正则表达式搜索代码内容
- 需要进行文件发现的大型代码库
Configuration options:
- **root_path**  `str` `required`要搜索的根目录。所有文件操作均相对于此路径。
- **use_ripgrep**  `bool` `default:"True"`是否使用 ripgrep 进行搜索。如果 ripgrep 不可用，则回退到 Python 正则表达式。
- **max_file_size_mb**  `intdefault:"10"`搜索文件的最大大小（以 MB 为单位）。大于此值的文件将被跳过。
该中间件为代理添加了两个搜索工具：

Glob 工具 - 快速文件模式匹配：
- 支持类似 ** /*.py 、 src/**/*.ts 这样的模式
- 返回按修改时间排序的匹配文件路径
Grep 工具 - 使用正则表达式进行内容搜索：
- 完全支持正则表达式语法
- 按文件模式筛选， include 参数
- 三种输出模式： files_with_matches 、 content 、 count

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import FilesystemFileSearchMiddleware
from langchain.messages import HumanMessage


agent = create_agent(
    model,
    tools=[],
    middleware=[
        FilesystemFileSearchMiddleware(
            root_path="E:\Desktop\Langgraph\Langchain-hollow\src",
            use_ripgrep=True,
            max_file_size_mb=10,
        ),
    ],
)

# Agent can now use glob_search and grep_search tools
result = agent.invoke({
    "messages": [HumanMessage("看看有什么文件")]
})
result
# The agent will use:
# 1. glob_search(pattern="**/*.py") to find Python files
# 2. grep_search(pattern="async def", include="*.py") to find async functions

### Filesystem middleware  文件系统中间件
上下文工程是构建高效智能体的主要挑战之一。当使用返回可变长度结果的工具（例如 web_search 和 RAG）时，这一点尤其困难，因为过长的结果会迅速填满上下文窗口。

Deep Agents 的 FilesystemMiddleware 提供了四个用于与短期记忆和长期记忆交互的工具：
- `ls` ：列出文件系统中的文件
- `read_file` ：读取整个文件或文件中指定数量的行
- `write_file` ：向文件系统写入新文件
- `edit_file` ：编辑文件系统中已存在的文件

#### Short-term vs. long-term filesystem 短期文件系统与长期文件系统
默认情况下，这些工具会将数据写入图状态中的本地“文件系统”。要启用跨线程的持久存储，请配置一个 CompositeBackend ，将特定路径（例如 /memories/ ）路由到 StoreBackend 。

In [ ]:
from langchain.agents import create_agent
from deepagents.middleware import FilesystemMiddleware
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

agent = create_agent(
    model="claude-sonnet-4-6",
    store=store,
    middleware=[
        FilesystemMiddleware(
            backend=lambda rt: CompositeBackend(
                default=StateBackend(rt),
                routes={"/memories/": StoreBackend(rt)}
            ),
            custom_tool_descriptions={
                "ls": "Use the ls tool when...",
                "read_file": "Use the read_file tool to..."
            }  # Optional: Custom descriptions for filesystem tools
        ),
    ],
)

当您为 /memories/ 配置带有 StoreBackend ` CompositeBackend` 时，所有以 `/memories/` 为前缀的文件都会保存到持久存储中，并在不同的线程之间保留。不带此前缀的文件则保留在临时状态存储中。

### Subagent  次级代理人
将任务交给子代理可以隔离上下文，保持主（监督）代理的上下文窗口干净，同时还能深入执行任务。

Deep Agents 的子代理中间件允许您通过 task 工具提供子代理。

In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent
from deepagents.middleware.subagents import SubAgentMiddleware


@tool
def get_weather(city: str) -> str:
    """Get the weather in a city."""
    return f"The weather in {city} is sunny."

agent = create_agent(
    model,
    middleware=[
        SubAgentMiddleware(
            default_model=model,
            default_tools=[],
            subagents=[
                {
                    "name": "weather",
                    "description": "This subagent can get weather in cities.",
                    "system_prompt": "Use the get_weather tool to get the weather in a city.",
                    "tools": [get_weather],
                    "model": model,
                    "middleware": [],
                }
            ],
        )
    ],
)
res = agent.invoke({"messages": [HumanMessage("石家庄天气怎么样？")]})
res

子代理由名称 、 描述 、 系统提示和工具定义。您还可以为子代理提供自定义模型或额外的中间件 。当您希望为子代理提供一个与主代理共享的额外状态键时，这将特别有用。

In [ ]:
from langchain.agents import create_agent
from deepagents.middleware.subagents import SubAgentMiddleware
from deepagents import CompiledSubAgent
from langgraph.graph import StateGraph

# Create a custom LangGraph graph
def create_weather_graph():
    workflow = StateGraph(...)
    # Build your custom graph
    return workflow.compile()

weather_graph = create_weather_graph()

# Wrap it in a CompiledSubAgent
weather_subagent = CompiledSubAgent(
    name="weather",
    description="This subagent can get weather in cities.",
    runnable=weather_graph
)

agent = create_agent(
    model="claude-sonnet-4-6",
    middleware=[
        SubAgentMiddleware(
            default_model="claude-sonnet-4-6",
            default_tools=[],
            subagents=[weather_subagent],
        )
    ],
)

除了用户自定义的子代理之外，主代理始终可以访问一个 general-purpose 代理。该子代理拥有与主代理相同的指令以及所有可用的工具。 general-purpose 子代理的主要目的是实现上下文隔离——主代理可以将复杂的任务委托给该子代理，并获得简洁的答案，而无需调用中间工具。